[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [PyTorch and TensorFlow, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/pytorch-and-tensorflow-deep-dive.html)

# Batches and Data Pipelines &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

In [1]:
import contextlib
import os

import numpy as np
import torch
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Dataset, TensorDataset


@contextlib.contextmanager
def c_messages_hidden():
    """Send what C and C++ code writes to the error stream to os.devnull while the block runs."""
    saved = os.dup(2)
    devnull = os.open(os.devnull, os.O_WRONLY)
    os.dup2(devnull, 2)
    try:
        yield
    finally:
        os.dup2(saved, 2)
        os.close(devnull)
        os.close(saved)


os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
with c_messages_hidden():
    import tensorflow as tf
    import keras


def station_network_arrays(n=1200, seed=2025):
    """The made-up stations of the scikit-learn guide, as arrays: four features and two targets."""
    rng = np.random.default_rng(seed)
    latitude = rng.uniform(58.0, 71.0, n)
    longitude = 5 + 1.5 * (latitude - 58) + rng.uniform(0, 5, n)
    coast_km = np.minimum(rng.exponential(25, n), 250)
    elevation = np.clip(rng.gamma(1.5, 150, n) + 3 * coast_km, 0, 1800)
    shift = np.where(coast_km < 3, 0.8,
            np.where(elevation > 700, -0.5,
            np.where((coast_km < 30) & (rng.uniform(size=n) < 0.6), 0.3, 0.0)))
    rng.choice(4, n, p=[0.4, 0.3, 0.1, 0.2])   # each station's owner, which this guide leaves out
    rng.integers(1950, 2025, n)                 # and the year it was installed
    jan_mean = (1.5 - 0.55 * (latitude - 58) - 0.006 * elevation - 4.0 * (1 - np.exp(-coast_km / 40))
                + shift + rng.normal(0, 0.9, n))
    may_mean = jan_mean + 8.5 + 0.004 * coast_km + rng.normal(0, 1.2, n)
    features = np.column_stack([latitude.round(2), longitude.round(2), elevation.round(0), coast_km.round(1)])
    return features.astype("float32"), jan_mean.round(1).astype("float32"), (may_mean < 2.5).astype("float32")


FEATURES = ["latitude", "longitude", "elevation_m", "coast_km"]
features, jan_mean, snow_on_may_1 = station_network_arrays()
order = np.random.default_rng(0).permutation(len(features))
train_rows, valid_rows, test_rows = order[:800], order[800:1000], order[1000:]
mean, std = features[train_rows].mean(axis=0), features[train_rows].std(axis=0)
scaled = (features - mean) / std

print("torch", torch.__version__.split("+")[0], "| tensorflow", tf.__version__, "| keras", keras.__version__,
      "|", len(features), "stations")
keras.utils.set_random_seed(0)

torch 2.11.0 | tensorflow 2.21.0 | keras 3.13.2 | 1200 stations


The worked examples' tensors and dataset, rebuilt here so that each answer runs on its own.

In [2]:
X_train = torch.from_numpy(scaled[train_rows])
y_train = torch.from_numpy(jan_mean[train_rows]).unsqueeze(1)
X_test = torch.from_numpy(scaled[test_rows])
y_test = torch.from_numpy(jan_mean[test_rows]).unsqueeze(1)
train_set = TensorDataset(X_train, y_train)

**1.** Make a `DataLoader` of `train_set` with `batch_size=100`, and print how many batches it makes and the size of each.

In [3]:
loader = DataLoader(train_set, batch_size=100)
print(len(loader), "batches:", [len(xb) for xb, yb in loader])

8 batches: [100, 100, 100, 100, 100, 100, 100, 100]


800 stations make eight batches of exactly 100, so none is short and `drop_last` would change nothing.

**2.** Print the first batch of eight row numbers for three epochs from a shuffling `DataLoader` of `row_numbers` whose generator is seeded with 1.

In [4]:
row_numbers = TensorDataset(torch.arange(800))
shuffled = DataLoader(row_numbers, batch_size=8, shuffle=True, generator=torch.Generator().manual_seed(1))
for epoch in range(3):
    (first_batch,) = next(iter(shuffled))
    print("epoch", epoch, "starts with", first_batch.tolist())

epoch 0 starts with [524, 264, 495, 72, 63, 796, 423, 406]
epoch 1 starts with [595, 274, 239, 341, 743, 554, 746, 676]
epoch 2 starts with [490, 72, 280, 356, 113, 241, 477, 782]


Seed 1 gives a different sequence of orders from seed 0, and each epoch still starts with a different batch.

**3.** Write a `Dataset` class whose examples are a station's standardized features and its snow label as a one-element float tensor, and print the shapes of one batch of 16.

In [5]:
class SnowDataset(Dataset):
    """Stations by row number: standardized features and whether there was snow on 1 May."""

    def __init__(self, rows):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        row = self.rows[i]
        return torch.from_numpy(scaled[row]), torch.tensor([snow_on_may_1[row]])


features_batch, labels_batch = next(iter(DataLoader(SnowDataset(train_rows), batch_size=16)))
print(features_batch.shape, labels_batch.shape, labels_batch.dtype)

torch.Size([16, 4]) torch.Size([16, 1]) torch.float32


The labels come out as a (16, 1) tensor of 32-bit floats, the shape and type that `BCEWithLogitsLoss` expects, as the **Losses and Optimizers** notebook showed.

**4.** Use `train_with_batches` to train with batches of 32 and of 128, and compare their steps and losses with the three printed above.

In [6]:
def train_with_batches(batch_size, epochs=20):
    """Train a fresh 4-32-1 network on batches of the training stations; return steps and losses."""
    torch.manual_seed(0)
    net = torch.nn.Sequential(torch.nn.Linear(4, 32), torch.nn.ReLU(), torch.nn.Linear(32, 1))
    optimizer = torch.optim.Adam(net.parameters(), lr=0.01)
    loader = DataLoader(train_set, batch_size=batch_size, shuffle=True,
                        generator=torch.Generator().manual_seed(0))
    steps = 0
    for epoch in range(epochs):
        for xb, yb in loader:
            optimizer.zero_grad()
            torch.nn.functional.mse_loss(net(xb), yb).backward()
            optimizer.step()
            steps += 1
    with torch.no_grad():
        train_loss = torch.nn.functional.mse_loss(net(X_train), y_train).item()
        test_loss = torch.nn.functional.mse_loss(net(X_test), y_test).item()
    return steps, round(train_loss, 3), round(test_loss, 3)


for batch_size in (32, 128):
    steps, train_loss, test_loss = train_with_batches(batch_size)
    print(f"batch size {batch_size:3}: {steps:4} steps, training loss {train_loss}, test loss {test_loss}")

batch size  32:  500 steps, training loss 0.808, test loss 0.948
batch size 128:  140 steps, training loss 1.039, test loss 1.13


Batches of 32 make 500 steps and reach a test loss of 0.948, and batches of 128 make 140 steps and reach 1.13. Sizes 16, 32 and 64 end within 0.04 of each other, a difference one seed cannot settle. Choosing among them is the job of the validation stations, which the **Training Loops** notebook uses.

**5.** Build a `tf.data` pipeline from the raw features of the training stations that standardizes them with `map`, shuffles with `seed=1` and batches by 32, train the Keras network on it for 20 epochs from `keras.utils.set_random_seed(0)`, and print the test loss.

In [7]:
raw_tf = tf.data.Dataset.from_tensor_slices((features[train_rows], jan_mean[train_rows][:, None]))
pipeline = raw_tf.map(lambda x, y: ((x - mean) / std, y)).shuffle(800, seed=1).batch(32)

keras.utils.set_random_seed(0)
net_keras = keras.Sequential([keras.Input((4,)), keras.layers.Dense(32, activation="relu"), keras.layers.Dense(1)])
net_keras.compile(optimizer=keras.optimizers.Adam(0.01), loss="mse")
net_keras.fit(pipeline, epochs=20, verbose=0)
print("test loss", round(net_keras.evaluate(scaled[test_rows], jan_mean[test_rows], verbose=0), 3))

test loss 0.959


The test loss is 0.959, against 0.961 for the worked example's pipeline with batches of 64 and seed 0.

---

&#8592; **Back to:** [Batches and Data Pipelines](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pytorch-and-tensorflow-deep-dive/09-batches-and-data-pipelines.ipynb)  &nbsp;&middot;&nbsp;  [PyTorch and TensorFlow, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/pytorch-and-tensorflow-deep-dive.html)
